# 01. 이미지 분류 — Keras로 시작하는 딥러닝

TensorFlow의 고수준 API인 **Keras** 로 이미지 분류 모델을 만듭니다.
딥러닝의 표준 워크플로(데이터 준비 → 모델 정의 → 학습 → 평가)를 Keras의 간결한 문법으로 익힙니다.

데이터는 **Fashion-MNIST** 를 사용합니다. 10종류 의류(티셔츠, 바지, 신발 등)의 28×28 흑백 이미지로,
딥러닝 입문의 표준 데이터셋입니다. 실행 시 자동으로 다운로드됩니다.

1. **데이터 준비** — Fashion-MNIST 로드 및 시각화
2. **모델 정의** — Keras로 CNN 구성
3. **학습** — `model.fit` 으로 학습하고 학습 곡선 확인
4. **평가** — 정확도 측정 및 예측 결과 시각화

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. 데이터 준비

`keras.datasets.fashion_mnist` 로 데이터를 불러옵니다. 픽셀값(0~255)을 0~1로 정규화하면 학습이 안정됩니다.
CNN 입력 형식에 맞게 채널 차원(흑백=1)을 추가합니다.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

x_train = (x_train / 255.0).astype('float32')[..., None]  # (N,28,28) -> (N,28,28,1)
x_test = (x_test / 255.0).astype('float32')[..., None]

class_names = ['티셔츠', '바지', '풀오버', '드레스', '코트',
               '샌들', '셔츠', '스니커즈', '가방', '앵클부츠']
print('학습:', x_train.shape, '| 테스트:', x_test.shape)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i].squeeze(), cmap='gray')
    ax.set_title(class_names[y_train[i]])
    ax.axis('off')
plt.suptitle('Fashion-MNIST 샘플', fontsize=14)
plt.tight_layout(); plt.show()

## 2. 모델 정의

Keras `Sequential` API로 CNN을 쌓습니다. 합성곱·풀링으로 특징을 추출하고, 완전연결 층으로 10개 클래스를 분류합니다.
`model.summary()` 로 각 층의 출력 형태와 파라미터 수를 확인할 수 있습니다.

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10, activation='softmax'),
])
model.summary()

## 3. 학습

- **손실 함수**: `sparse_categorical_crossentropy` (정수 라벨 분류용)
- **옵티마이저**: Adam
- `model.fit` 은 학습 기록(history)을 반환하며, 여기서 에폭별 정확도·손실을 얻을 수 있습니다.
- 검증 데이터(`validation_split`)로 매 에폭 일반화 성능을 함께 확인합니다.

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(x_train, y_train, epochs=10, batch_size=128,
                    validation_split=0.1, verbose=2)

## 4. 평가

테스트셋 정확도를 측정하고, 학습 곡선과 예측 결과를 시각화합니다. CNN은 보통 **90% 안팎**의 정확도를 보입니다.

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'테스트 정확도: {test_acc*100:.2f}%')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(history.history['accuracy'], label='학습')
ax1.plot(history.history['val_accuracy'], label='검증')
ax1.set_title('정확도'); ax1.set_xlabel('epoch'); ax1.legend()
ax2.plot(history.history['loss'], label='학습')
ax2.plot(history.history['val_loss'], label='검증')
ax2.set_title('손실'); ax2.set_xlabel('epoch'); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
preds = model.predict(x_test[:10], verbose=0).argmax(1)
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_test[i].squeeze(), cmap='gray')
    ok = preds[i] == y_test[i]
    ax.set_title(f'{class_names[preds[i]]}', color='black' if ok else 'red')
    ax.axis('off')
plt.suptitle('예측 결과 (빨강=오답)', fontsize=14)
plt.tight_layout(); plt.show()

### 정리

- Keras `Sequential` API로 CNN을 정의하고 `compile` → `fit` → `evaluate` 의 표준 워크플로를 익혔습니다.
- `history` 로 학습/검증 곡선을 그려 학습 과정을 진단했습니다.
- 다음 노트북에서는 **사전학습 모델(MobileNetV2)을 활용한 전이학습**을 다룹니다.